# DistilBERT Fine-Tuning — Google Colab

**UPLOAD AND RUN THIS ON GOOGLE COLAB — NOT LOCALLY**

**Project:** MSc AI Dissertation — AI-Generated Text Detection  
**Student:** Abdul Hannaan Mohammed | B00409227 | UWS  

**Files to upload when prompted:**
- `distilbert_train.csv` (from data/processed/)
- `distilbert_val.csv` (from data/processed/)

**Output:** `distilbert-hc3-best.zip` — download and unzip into `models/checkpoints/`

**Note:** DistilBERT is 40% smaller and 60% faster than BERT.  
Expected training time: **~50–60 minutes on T4 GPU**

**Before running:** Runtime > Change runtime type > T4 GPU

## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUTPUT = '/content/drive/MyDrive/MSc-AI-Detection/models'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f'Drive mounted. Output: {DRIVE_OUTPUT}')

## Cell 2 — Upload CSV Files

Upload `distilbert_train.csv` and `distilbert_val.csv` from your local `data/processed/` folder.

In [ ]:
from google.colab import files
import pandas as pd
import io

print('Select distilbert_train.csv and distilbert_val.csv together...')
uploaded = files.upload()

train_df = pd.read_csv(io.BytesIO(uploaded['distilbert_train.csv']))
val_df   = pd.read_csv(io.BytesIO(uploaded['distilbert_val.csv']))

print(f'Train : {len(train_df):,} rows | Human: {(train_df["label"]==0).sum():,}  AI: {(train_df["label"]==1).sum():,}')
print(f'Val   : {len(val_df):,} rows  | Human: {(val_df["label"]==0).sum():,}   AI: {(val_df["label"]==1).sum():,}')

## Cell 3 — Check GPU

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU — go to Runtime > Change runtime type > T4 GPU')

## Cell 4 — Install Libraries

In [ ]:
!pip install -q transformers accelerate evaluate
print('Done.')

## Cell 5 — Seeds and Hyperparameters

In [ ]:
import random
import numpy as np
from transformers import set_seed

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

MODEL_NAME    = 'distilbert-base-uncased'
NUM_LABELS    = 2
MAX_LENGTH    = 512
BATCH_SIZE    = 16   # DistilBERT is smaller so we can use larger batch
GRAD_ACCUM    = 2    # effective batch = 32
NUM_EPOCHS    = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.1

print(f'Model: {MODEL_NAME}')
print(f'Batch size: {BATCH_SIZE} (effective: {BATCH_SIZE*GRAD_ACCUM})')
print(f'Epochs: {NUM_EPOCHS}')

## Cell 6 — Tokenise and Build Datasets

In [ ]:
from torch.utils.data import Dataset
from transformers import DistilBertTokenizerFast

class TextDataset(Dataset):
    """PyTorch Dataset for DistilBERT — no token_type_ids needed."""
    def __init__(self, df, tokeniser, max_length):
        self.labels    = df['label'].tolist()
        self.encodings = tokeniser(
            df['text'].tolist(),
            max_length=max_length,
            padding='max_length',
            truncation=True
        )
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return {
            'input_ids':      torch.tensor(self.encodings['input_ids'][idx],      dtype=torch.long),
            'attention_mask': torch.tensor(self.encodings['attention_mask'][idx], dtype=torch.long),
            'labels':         torch.tensor(self.labels[idx],                      dtype=torch.long)
        }

print('Loading DistilBERT tokeniser...')
tokeniser = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

print('Tokenising (takes ~30 seconds)...')
train_dataset = TextDataset(train_df, tokeniser, MAX_LENGTH)
val_dataset   = TextDataset(val_df,   tokeniser, MAX_LENGTH)

print(f'Train: {len(train_dataset):,} | Val: {len(val_dataset):,}')

## Cell 7 — Load DistilBERT Model

In [ ]:
from transformers import DistilBertForSequenceClassification

print('Loading distilbert-base-uncased (~265MB)...')
model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {n_params/1e6:.1f}M  (40% smaller than BERT)')

## Cell 8 — Train

Expected time: **~50–60 minutes on T4 GPU**

> SCREENSHOT: Screenshot the training progress.  
> Save as: `results/screenshots/07_distilbert_training_curves.png`

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    """Accuracy, Precision, Recall, F1 after each epoch."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc   = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', pos_label=1)
    return {'accuracy': round(acc,4), 'precision': round(p,4), 'recall': round(r,4), 'f1': round(f1,4)}

training_args = TrainingArguments(
    output_dir='/content/distilbert-hc3',
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    fp16=True,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=50,
    report_to='none',
    seed=SEED,
    dataloader_num_workers=2,
    save_total_limit=1,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print('Starting DistilBERT training...')
train_result = trainer.train()
print(f'\nTraining complete. Time: {train_result.metrics["train_runtime"]/60:.1f} minutes')

## Cell 9 — Print Final Results

> SCREENSHOT: Screenshot this output.  
> Save as: `results/screenshots/07_distilbert_training_curves.png`

In [ ]:
history   = trainer.state.log_history
eval_logs = [h for h in history if 'eval_f1' in h]
best      = max(eval_logs, key=lambda x: x['eval_f1'])

print('=== DISTILBERT BEST VALIDATION RESULTS ===')
print(f'  Epoch     : {best["epoch"]}')
print(f'  Accuracy  : {best["eval_accuracy"]:.4f}')
print(f'  Precision : {best["eval_precision"]:.4f}')
print(f'  Recall    : {best["eval_recall"]:.4f}')
print(f'  F1        : {best["eval_f1"]:.4f}')

## Cell 10 — Save Model to Drive and Download

After downloading, unzip and place the folder at:  
`models/checkpoints/distilbert-hc3-best/`

In [ ]:
import shutil

SAVE_PATH = '/content/distilbert-hc3-best'
trainer.save_model(SAVE_PATH)
tokeniser.save_pretrained(SAVE_PATH)
print(f'Model saved to: {SAVE_PATH}')

ZIP_PATH = '/content/distilbert-hc3-best.zip'
shutil.make_archive('/content/distilbert-hc3-best', 'zip', '/content', 'distilbert-hc3-best')
print(f'Zipped: {ZIP_PATH}')

shutil.copy(ZIP_PATH, os.path.join(DRIVE_OUTPUT, 'distilbert-hc3-best.zip'))
print('Backup saved to Drive.')

files.download(ZIP_PATH)
print('Download started. Unzip and place at: models/checkpoints/distilbert-hc3-best/')